# G/F 16_2 · Comparación a lazo abierto de las reglas de crecimiento

Ambas simulaciones usan los parámetros químicos guardados del 16_2, sus pulsos, sus entradas de temperatura, sus condiciones iniciales y su capa de CO₂. La variante del EKF 16 permite que el crecimiento y la captación de YAN continúen mientras quede glucosa o fructosa. Cada azúcar agotado deja de aportar al crecimiento. Se compara una sola modificación de la dinámica en LAB004–009 y LAB016–024.

In [ ]:
from pathlib import Path
from types import FunctionType
import json

import nbformat
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/wine_fermentation").is_dir())
source, = (ROOT / "notebooks").glob("16_2_*.source.ipynb")
calibration = nbformat.read(source, 4)
saved = nbformat.read(source.with_name(source.name.replace(".source.", ".executed.")), 4)
assert [cell.source for cell in calibration.cells] == [cell.source for cell in saved.cells]
[output] = [o.text.split("PARÁMETROS QUÍMICOS COMPLETOS", 1)[1].lstrip()
            for o in saved.cells[5].outputs
            if o.output_type == "stream" and "PARÁMETROS QUÍMICOS COMPLETOS" in o.text]
theta, _ = json.JSONDecoder().raw_decode(output)

def reference(index, start=None, stop=None):
    code = calibration.cells[index].source
    if start is not None:
        assert code.count(start) == 1
        code = code[code.index(start):]
    if stop is not None:
        assert code.count(stop) == 1
        code = code[:code.index(stop)]
    return code

setup = {"__name__": "__main__", "display": lambda *_: None}
for code in (reference(1, stop="start=kin.normalize_parameters"),
             reference(3),
             reference(12, stop="cores,caches,activation=build_curves(theta)"),
             reference(12, start="KLA, GAIN =", stop="curves={lab:gas(lab,KLA,GAIN)"),
             reference(13)):
    exec(compile(code, source.name, "exec"), setup)
kin = setup["kin"]
assert set(theta) == set(kin.PARAMETERS)
display(pd.Series(theta, name="Parámetros guardados del 16_2"))
print("Capa de CO₂ del 16_2", setup["HOLDOUT_GAS_PARAMS"])

## Integración y regla de crecimiento

Se localiza cuándo cada azúcar llega a cero y se integra de nuevo desde ese punto. El mismo manejo de eventos se usa en ambas simulaciones. En la producción de CO₂, el aporte de crecimiento sigue la misma regla que los balances químicos.

In [ ]:
BASE = "Regla del 16_2"
AVAILABLE = "Regla del EKF 16"
LABELS = (BASE, AVAILABLE)
SUGARS = np.array([2, 3])
solve_ivp = setup["solve_ivp"]

def integrate_sugar_events(rhs, interval, initial, *, t_eval, **options):
    start, end = interval
    state = np.array(initial, copy=True)
    active = list(SUGARS[state[SUGARS] > 0.])
    pieces_t, pieces_y = [], []
    remaining = np.asarray(t_eval)
    while start < end:
        frozen = list(set(SUGARS) - set(active))

        def stage_rhs(t, y):
            physical = y.copy()
            physical[active] = np.maximum(physical[active], np.finfo(float).tiny)
            physical[frozen] = 0.
            derivative = rhs(t, physical)
            derivative[frozen] = 0.
            return derivative

        events = []
        for index in active:
            def depleted(t, y, index=index):
                return y[index]
            depleted.terminal = True
            depleted.direction = -1
            events.append(depleted)

        result = solve_ivp(stage_rhs, (start, end), state, t_eval=remaining,
                           events=events or None, **options)
        if not result.success:
            return result
        pieces_t.append(np.asarray(result.t))
        pieces_y.append(np.asarray(result.y).reshape(len(initial), -1))
        if result.status != 1:
            break
        [hit] = [i for i, times in enumerate(result.t_events) if len(times)]
        start = result.t_events[hit][0]
        state = result.y_events[hit][0].copy()
        state[active.pop(hit)] = 0.
        remaining = remaining[remaining > start]

    result.t = np.concatenate(pieces_t)
    result.y = np.concatenate(pieces_y, axis=1)
    np.testing.assert_array_equal(result.t, t_eval)
    return result

# Mantener el integrador, los pulsos y los tiempos de evaluación del 16_2.
simulate_source = reference(3, start="def simulate(b,p):", stop="def with_jumps")
old_guard = simulate_source[simulate_source.index("  if y[2]<=0 or y[3]<=0:"):
                            simulate_source.index("  return out")]
new_guard = '''  if y[2]<=0 or y[3]<=0:
   q=kin.kinetic_terms(p,kin.temperature_at(b,t),*y);growth=q['mu']*max(float(y[0]),0)
   if y[2]<=0 and y[3]<=0:
    out[0]-=growth;out[1]+=growth/p['Yxn']
   if y[2]<=0:
    out[2]+=growth/p['Yxg']
   if y[3]<=0:
    out[3]+=growth/p['Yxf']
'''
holdout_source = reference(14, start="def run_holdout(theta):",
                           stop="holdout_kinetic, holdout_plot, holdout_gas, holdout_mean_gas = run_holdout(theta)")
historical_source = reference(12, start="def build_curves(parameters):", stop="PCO2_BAR=1.0")

training = {batch.batch: batch for batch in setup["treated"]}
historical = setup["batches"]
recent = setup["holdout_batches"]
batches = {**training, **historical, **recent}
assert set(batches) == {f"LAB{n:03}" for n in (*range(4, 10), *range(16, 25))}
trajectories, gas_curves, recent_gas, training_raw = {}, {}, {}, {}

for label in LABELS:
    context = {**setup, "solve_ivp": integrate_sugar_events}
    for name in ("configure_gas", "gas"):
        function = setup[name]
        context[name] = FunctionType(function.__code__, context, argdefs=function.__defaults__)
    chemistry, history, holdout = simulate_source, historical_source, holdout_source
    if label == AVAILABLE:
        assert chemistry.count(old_guard) == 1
        chemistry = chemistry.replace(old_guard, new_guard)
        history = history.replace("y[2]<=0 or y[3]<=0", "y[2]<=0 and y[3]<=0")
        holdout = holdout.replace("y[2] > 0 and y[3] > 0", "y[2] > 0 or y[3] > 0")

    exec(compile(chemistry + history + holdout, source.name, "exec"), context)
    context["simulate_holdout"] = context["simulate"]
    training_raw[label] = {lab: context["simulate"](batch, theta) for lab, batch in training.items()}
    cores, caches, activation = context["build_curves"](theta)
    context.update(caches=caches, activation=activation)
    context["configure_gas"](cores)
    gas_curves[label] = {lab: context["gas"](lab, *setup["HOLDOUT_GAS_PARAMS"]) for lab in historical}
    holdout_cores, _, holdout_gas, recent_gas[label] = context["run_holdout"](theta)
    gas_curves[label].update(holdout_gas)
    trajectories[label] = {**training_raw[label], **cores, **holdout_cores}

# Verificar condiciones iniciales, pulsos y mallas comunes.
for lab, batch in batches.items():
    a, b = (trajectories[label][lab] for label in LABELS)
    np.testing.assert_array_equal(a.index, b.index)
    np.testing.assert_array_equal(a.iloc[0], b.iloc[0])
    for frame in (a, b):
        assert (frame.to_numpy() >= 0.).all()
        received = [(j["t"], j["after"][1] - j["before"][1]) for j in frame.attrs["pulse_jumps"]]
        np.testing.assert_allclose(received, batch.pulses["N"], atol=1e-12, rtol=0.)
    # Las curvas deben coincidir antes de que se agote el primer azúcar.
    both = (a.G > 0.) & (a.F > 0.)
    stops = np.flatnonzero(~both.to_numpy())
    last = stops[0] if len(stops) else len(a)
    np.testing.assert_allclose(a.iloc[:last], b.iloc[:last], atol=2e-7, rtol=2e-6)

print(f"{len(batches)} lotes y {len(batches) * len(LABELS)} simulaciones químicas verificadas.")

## Residuos y agotamiento de azúcar

En LAB004–009 se usa el soporte y la ponderación del objetivo histórico. Los RMSE químicos excluyen la condición inicial. Para LAB016–018, la referencia de G+F procede de densidad. En LAB022–024, el flujo de CO₂ se compara en medias de 30 minutos y conserva los huecos de la medición.

In [ ]:
VARS = {
    "X": ("Biomasa viable", "g/L", 1.), "N": ("YAN", "mg N/L", 1000.),
    "G": ("Glucosa", "g/L", 1.), "F": ("Fructosa", "g/L", 1.),
    "GF": ("G+F", "g/L", 1.), "E": ("Etanol", "g/L", 1.),
    "CO2": ("Flujo de CO₂", "g/(L·h)", 1.),
}
recent_columns = {"X": ("holdout_biomass", "X_g_L"), "N": ("holdout_chemistry", "YAN for plot"),
                  "G": ("holdout_chemistry", "GLUCOSE-320"), "F": ("holdout_chemistry", "Fructose (derived)"),
                  "GF": ("holdout_chemistry", "GLU-FRU-320")}
v = setup["v"]

def observed(lab, state):
    batch = batches[lab]
    factor = VARS[state][2]
    horizon = v.horizons[lab] if lab in historical else batch.time[-1]
    if lab in training:
        if state == "CO2":
            return pd.Series(dtype=float)
        values = batch.observations["G"] + batch.observations["F"] if state == "GF" else batch.observations[state]
        times = batch.time
        values = factor * values
    elif state == "CO2":
        if lab in recent:
            common_windows = recent_gas[BASE][lab].index
            return setup["holdout_flow"][lab].reindex(common_windows).dropna()
        frame = setup["obs"][lab]
        times, values = frame.time_h.to_numpy(), frame.co2_raw_g_l_h.to_numpy()
    elif lab in recent:
        if state not in recent_columns:
            return pd.Series(dtype=float)
        table, column = recent_columns[state]
        frame = setup[table]
        samples = frame.loc[frame.lab.eq(lab) & frame["sample"].ne("01")]
        times, values = samples.hours.to_numpy(), samples[column].to_numpy()
    elif lab in setup["TRAIN"]:
        frame = setup["chemical_observations"](v.chem, lab, state)
        times, values = frame.t_h.to_numpy(), factor * frame[setup["CHEMICAL_COLUMNS"][state]].to_numpy()
    elif state == "X":
        frame = v.biomass.loc[v.biomass.lab.eq(lab)]
        times, values = frame.t_h.to_numpy(), frame.X_viable_g_l.to_numpy()
    elif state == "GF":
        frame = v.density.loc[v.density.lab.eq(lab)]
        times, values = frame.t_h.to_numpy(), 2.5 * (frame.density_g_l.to_numpy() - 1000.)
    else:
        return pd.Series(dtype=float)
    mask = np.isfinite(values) & (times > batch.time[0]) & (times <= horizon)
    return pd.Series(values[mask], index=times[mask]).sort_index()

def predicted(label, lab, state, *, plot=False):
    if state == "CO2":
        return recent_gas[label][lab] if lab in recent else gas_curves[label][lab]["qpred"]
    frame = trajectories[label][lab]
    if plot:
        frame = setup["with_jumps"](frame)
    return VARS[state][2] * frame[state]

samples = {(lab, state): observed(lab, state) for lab in batches for state in VARS}
rows = []
for (lab, state), obs in samples.items():
    if obs.empty:
        continue
    row = {"LAB": lab, "Variable": state, "n": len(obs)}
    for label in LABELS:
        pred = predicted(label, lab, state)
        errors = np.interp(obs.index, pred.index, pred.values) - obs.values
        row[label] = np.sqrt(np.mean(errors**2))
    row["Δ RMSE"] = row[AVAILABLE] - row[BASE]
    rows.append(row)
rmse = pd.DataFrame(rows).set_index(["LAB", "Variable"])
display(rmse.round(4))

weighted = {}
for label in LABELS:
    residuals = [(trajectories[label][b.batch][state].to_numpy()[mask] - b.observations[state][mask])
                 / setup["sigma"](state, b.observations[state][mask])
                 for b, state, mask in setup["support"]]
    weighted[label] = np.mean(np.concatenate(residuals)**2)
print("MSE ponderado LAB004–009:", weighted)
np.testing.assert_allclose(weighted[BASE], 4.715502281177189, rtol=2e-5)

summary = []
for lab, batch in batches.items():
    a, b = (trajectories[label][lab] for label in LABELS)
    delta = (a - b).abs().max()
    row = {"LAB": lab, "Δ máximo X [g/L]": delta.X, "Δ máximo N [mg N/L]": 1000 * delta.N}
    for label, frame in zip(LABELS, (a, b)):
        first = frame.index[(frame.G <= 1e-8) | (frame.F <= 1e-8)]
        row[f"Primer azúcar agotado [h] · {label}"] = first[0] if len(first) else np.nan
        row[f"YAN final [mg N/L] · {label}"] = 1000 * frame.N.iloc[-1]
    summary.append(row)
summary = pd.DataFrame(summary).set_index("LAB")
display(summary.round(4))
print(summary.round(4).to_string())
print(rmse.loc[["LAB022", "LAB023", "LAB024"]].round(4).to_string())

## Trayectorias de los dos modelos

In [ ]:
styles = {BASE: dict(color="#737373", linestyle="--", zorder=3),
          AVAILABLE: dict(color="#0072B2", zorder=2)}
for group, labs in (("Calibración química", tuple(training)),
                    ("Evaluación del 16_2", (*historical, *recent))):
    for state, (title, unit, _) in VARS.items():
        if state == "CO2" and group == "Calibración química":
            continue
        fig, axes = plt.subplots(len(labs) // 3, 3, figsize=(14, 3.3 * len(labs) // 3 + .6),
                                 layout="constrained")
        for ax, lab in zip(axes.flat, labs):
            for label in LABELS:
                frame = predicted(label, lab, state, plot=True)
                ax.plot(frame.index, frame.values, label=label, **styles[label])
            obs = samples[lab, state]
            if state == "CO2":
                if lab in recent:
                    obs = setup["holdout_flow"][lab]
                ax.plot(obs.index, obs.values, color="black", alpha=.65, lw=.9, label="Medido")
            else:
                ax.scatter(obs.index, obs.values, c="black", s=13, zorder=5, label="Muestras")
            for pulse, _ in batches[lab].pulses["N"]:
                ax.axvline(pulse, c="gray", ls=":", lw=.8)
            subtitle = lab
            if (lab, state) in rmse.index:
                old, new = rmse.loc[(lab, state), list(LABELS)]
                subtitle += f" · RMSE {old:.3f} → {new:.3f}"
            horizon = v.horizons[lab] if lab in historical else batches[lab].time[-1]
            ax.set(title=subtitle, xlabel="Tiempo [h]", ylabel=f"{title} [{unit}]", xlim=(0, horizon))
            ax.grid(alpha=.2)
        axes.flat[0].legend(fontsize=8)
        fig.suptitle(f"{title} · {group}")
        plt.show()